# Part 8: Cross-Sectional Sector Analysis & Pooled Training

**Research Question:** Under rigorous temporal split, does chart-pattern predictive
power vary systematically across GICS market sectors?

**Framework:** Adaptive Markets Hypothesis (Lo, 2004) -- market efficiency varies
by segment and over time.

## Experiment Design

| Stage | Description | Key Output |
|-------|-------------|------------|
| **2** | Sector-pooled training (11 GICS sectors) | Per-sector accuracy, binomial tests, Bonferroni correction |
| **2.5** | Financial category grouping (3 categories) | Category comparison, volatility-performance correlation |
| **3** | Full-pooled training (all 1.3M+ images) | Mode-collapse check, specialist vs generalist comparison |

**Statistical Enhancements:** Permutation test, Bonferroni/FDR correction,
effect-size (Cohen's h) reporting.

---
**Config:** window=20, horizon=5, embargo=10 days, test >= 2023-01-01,
VGG16 fine-tune block5, XGBoost (64x64 flatten), 6-class return bins.

**Data:** 501 stocks, ~1.37M candlestick chart images (480x480 saved, loaded at 224x224).

**Runtime:** ~4-8 h on A100 GPU.

In [ ]:
# Cell 1: Imports & Setup
!pip install -q xgboost

import os, json, time, warnings, gc, pickle, hashlib
from pathlib import Path
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU available: {gpus[0].name}')
else:
    print('WARNING: No GPU detected. Training will be very slow.')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.multitest import multipletests

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense, Dropout,
                                     Input, Flatten)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import VGG16
from xgboost import XGBClassifier

# ---- Constants ----
CLASS_NAMES_6 = ['down_3plus', 'down_2_3', 'down_1_2',
                 'up_1_2', 'up_2_3', 'up_3plus']
NUM_CLASSES   = 6
IMG_SIZE      = 224
XGB_IMG_SIZE  = 64
TEST_START    = '2023-01-01'
EMBARGO_DAYS  = 10

FINANCIAL_CATEGORIES = {
    'Cyclical Growth': ['Information Technology', 'Consumer Discretionary',
                        'Communication Services'],
    'Cyclical Value':  ['Financials', 'Industrials', 'Materials', 'Energy'],
    'Defensive':       ['Consumer Staples', 'Health Care', 'Utilities',
                        'Real Estate'],
}
# Reverse mapping: sector -> category
SECTOR_TO_CATEGORY = {}
for cat, sectors in FINANCIAL_CATEGORIES.items():
    for s in sectors:
        SECTOR_TO_CATEGORY[s] = cat

print(f'TensorFlow {tf.__version__}')
print(f'Classes: {CLASS_NAMES_6}')
print('All imports OK')

In [ ]:
# Cell 2: Mount Drive + Unzip images.zip to local storage + Load metadata

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE   = Path('/content/drive/MyDrive/SP500_Chart_Dataset')
META_DIR     = DRIVE_BASE / 'metadata'
PRICE_DIR    = DRIVE_BASE / 'price_data'
RESULTS_DIR  = DRIVE_BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Unzip images to LOCAL storage for fast I/O ---
LOCAL_IMG_DIR = Path('/content/data/images')
ZIP_PATH      = DRIVE_BASE / 'images.zip'

if not LOCAL_IMG_DIR.exists() or len(list(LOCAL_IMG_DIR.glob('*'))) < 100:
    print('Unzipping images.zip to /content/data/ (this takes a few minutes)...')
    t0 = time.time()
    !unzip -q -o "{ZIP_PATH}" -d /content/data/
    # Handle case where zip contains images/ subdirectory or flat structure
    if not LOCAL_IMG_DIR.exists():
        # Try common alternative structures
        alt = Path('/content/data/cross_sectional_data/images')
        if alt.exists():
            !mv /content/data/cross_sectional_data/images /content/data/images
        else:
            raise FileNotFoundError(
                f'Could not find images directory after unzip. '
                f'Contents of /content/data/: {list(Path("/content/data").iterdir())}')
    n_dirs = len(list(LOCAL_IMG_DIR.glob('*')))
    print(f'Unzipped in {time.time()-t0:.0f}s  |  {n_dirs} ticker directories')
else:
    n_dirs = len(list(LOCAL_IMG_DIR.glob('*')))
    print(f'Images already unzipped: {n_dirs} ticker directories in {LOCAL_IMG_DIR}')

# --- Load metadata ---
with open(META_DIR / 'pipeline_summary.json') as f:
    summary = json.load(f)
with open(META_DIR / 'sp500_constituents.json') as f:
    constituents = json.load(f)

ok_tickers    = summary['ok_tickers']
ticker_sector = {c['ticker']: c['sector'] for c in constituents}
ticker_company = {c['ticker']: c['company'] for c in constituents}

# Build sector -> tickers mapping
sector_tickers = defaultdict(list)
for t in ok_tickers:
    sec = ticker_sector.get(t)
    if sec:
        sector_tickers[sec].append(t)

print(f'\nValid stocks: {len(ok_tickers)}')
print(f'Sectors: {len(sector_tickers)}')
print(f'Total samples: {summary["total_samples"]:,}')
print()
for sec in sorted(sector_tickers, key=lambda s: -len(sector_tickers[s])):
    print(f'  {sec:>30s}: {len(sector_tickers[sec]):>3d} stocks')

In [ ]:
# Cell 3: Data Loading Utilities  (sector pooling + tf.data pipeline)

# ------------------------------------------------------------------
# 3-A  Sample-level helpers
# ------------------------------------------------------------------

def load_stock_samples(ticker):
    """Load per-stock sample metadata from Drive."""
    p = META_DIR / f'samples_{ticker}.json'
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)


def _split_filter(samples, split):
    """Filter samples by temporal split.  Returns list of dicts."""
    ts  = pd.Timestamp(TEST_START)
    emb = pd.Timedelta(days=EMBARGO_DAYS)
    out = []
    for s in samples:
        ed = pd.Timestamp(s['end_date'])
        if split == 'train' and ed >= ts - emb:
            continue
        if split == 'test' and ed < ts:
            continue
        out.append(s)
    return out


def _image_path(ticker, sample):
    """Build local image path for a sample."""
    fname = f"{ticker}_{sample['index']}_{sample['end_date'].replace('-','')}.png"
    return LOCAL_IMG_DIR / ticker / sample['label'] / fname


# ------------------------------------------------------------------
# 3-B  Numpy loader  (for sector-pooled training, moderate data)
# ------------------------------------------------------------------

def load_sector_numpy(sector, split='train', img_size=IMG_SIZE, max_per_stock=None):
    """
    Load all images for a sector into numpy arrays.
    Returns (X, y, tickers_arr)  where tickers_arr tracks per-sample stock.
    """
    tickers = sector_tickers[sector]
    all_imgs, all_labels, all_tkrs = [], [], []

    for ticker in tickers:
        meta = load_stock_samples(ticker)
        if meta is None:
            continue
        samples = _split_filter(meta['samples'], split)
        if max_per_stock and len(samples) > max_per_stock:
            samples = samples[:max_per_stock]

        for s in samples:
            fpath = _image_path(ticker, s)
            if not fpath.exists():
                continue
            try:
                img = tf.keras.utils.load_img(str(fpath),
                                              target_size=(img_size, img_size))
                all_imgs.append(tf.keras.utils.img_to_array(img) / 255.0)
                all_labels.append(s['label_idx'])
                all_tkrs.append(ticker)
            except Exception:
                continue

    if not all_imgs:
        return None, None, None
    return (np.array(all_imgs, dtype=np.float32),
            np.array(all_labels, dtype=np.int32),
            np.array(all_tkrs))


# ------------------------------------------------------------------
# 3-C  tf.data pipeline  (for Stage 3 full-pooled, avoids OOM)
# ------------------------------------------------------------------

def _build_file_label_lists(tickers, split):
    """Collect (file_path_str, label_idx) for a list of tickers."""
    paths, labels, tkrs = [], [], []
    for ticker in tickers:
        meta = load_stock_samples(ticker)
        if meta is None:
            continue
        samples = _split_filter(meta['samples'], split)
        for s in samples:
            fp = _image_path(ticker, s)
            if fp.exists():
                paths.append(str(fp))
                labels.append(s['label_idx'])
                tkrs.append(ticker)
    return paths, labels, tkrs


def _parse_image(path, label):
    """tf.data map function: read + decode + resize + normalise."""
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def _parse_image_small(path, label):
    """tf.data map function for XGBoost: 64x64 flatten."""
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [XGB_IMG_SIZE, XGB_IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.reshape(img, [XGB_IMG_SIZE * XGB_IMG_SIZE * 3])
    return img, label


def make_tf_dataset(paths, labels, batch_size=128, shuffle=True,
                    parse_fn=_parse_image):
    """Create a tf.data.Dataset from file paths and labels."""
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 50000), seed=42)
    ds = ds.map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def compute_class_weights_from_labels(labels):
    """Compute balanced class weights dict."""
    classes = np.unique(labels)
    if len(classes) < 2:
        return None
    weights = compute_class_weight('balanced', classes=classes, y=np.array(labels))
    return dict(zip(classes.tolist(), weights.tolist()))


print('Data loading utilities ready')
print(f'  LOCAL_IMG_DIR = {LOCAL_IMG_DIR}')
print(f'  IMG_SIZE={IMG_SIZE}, XGB_IMG_SIZE={XGB_IMG_SIZE}')

In [ ]:
# Cell 4: Model Building Functions  (VGG16 fine-tune block5, XGBoost)

def build_vgg16_ft_block5(n_classes=NUM_CLASSES):
    """VGG16 with only block5 fine-tuned, GAP + Dense head."""
    base = VGG16(weights='imagenet', include_top=False,
                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
    for layer in base.layers:
        layer.trainable = 'block5' in layer.name
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(n_classes, activation='softmax'),
    ])
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def build_vgg16_feature_extractor():
    """VGG16 base outputting GAP features (for XGBoost 2nd-stage)."""
    base = VGG16(weights='imagenet', include_top=False,
                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    model = Model(inputs=base.input,
                  outputs=GlobalAveragePooling2D()(base.output))
    return model


def train_vgg16_on_arrays(X_train, y_train, X_test, y_test,
                          epochs=100, batch_size=64, patience=10):
    """
    Train VGG16 on numpy arrays.  Returns y_pred on test set.
    Used for Stage 2 (sector-pooled, moderate sizes).
    """
    tf.random.set_seed(42)
    np.random.seed(42)

    model = build_vgg16_ft_block5()
    cw = compute_class_weights_from_labels(y_train)
    if cw is None:
        return None, {}

    # Validation: last 15 % of training data (temporal order preserved)
    val_n = max(int(len(X_train) * 0.15), 32)
    X_val, y_val = X_train[-val_n:], y_train[-val_n:]
    X_tr,  y_tr  = X_train[:-val_n],  y_train[:-val_n]

    if len(X_tr) < 50:
        del model; tf.keras.backend.clear_session(); gc.collect()
        return None, {}

    history = model.fit(
        X_tr, y_tr, epochs=epochs, batch_size=batch_size,
        validation_data=(X_val, y_val), class_weight=cw,
        callbacks=[
            EarlyStopping('val_loss', patience=patience,
                          restore_best_weights=True, verbose=0),
            ReduceLROnPlateau('val_loss', factor=0.5, patience=3, verbose=0),
        ],
        verbose=0,
    )
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    info = {'epochs_run': len(history.history['loss']),
            'best_val_loss': float(min(history.history['val_loss']))}

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return y_pred, info


def train_vgg16_on_tfdata(ds_train, ds_val, class_weight,
                          epochs=50, patience=5):
    """
    Train VGG16 using tf.data pipeline (Stage 3, large data).
    Returns the trained model.
    """
    tf.random.set_seed(42)
    model = build_vgg16_ft_block5()
    history = model.fit(
        ds_train, epochs=epochs,
        validation_data=ds_val,
        class_weight=class_weight,
        callbacks=[
            EarlyStopping('val_loss', patience=patience,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau('val_loss', factor=0.5, patience=2, verbose=1),
        ],
        verbose=1,
    )
    return model, history


def train_xgboost_flat(X_train, y_train, X_test, y_test):
    """
    XGBoost on flattened 64x64 pixel features.  Returns y_pred.
    X arrays should already be 2-D (n_samples, 64*64*3).
    """
    cw = compute_class_weights_from_labels(y_train)
    if cw is None:
        return None
    sw = np.array([cw[y] for y in y_train])

    clf = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.01,
        objective='multi:softmax', num_class=NUM_CLASSES,
        random_state=42, use_label_encoder=False,
        eval_metric='mlogloss', tree_method='hist',
        n_jobs=-1,
    )
    clf.fit(X_train, y_train, sample_weight=sw, verbose=False)
    y_pred = clf.predict(X_test)
    del clf; gc.collect()
    return y_pred


# ------------------------------------------------------------------
# Statistical testing helpers
# ------------------------------------------------------------------

def eval_predictions(y_true, y_pred, majority_baseline_frac=None):
    """
    Compute accuracy, f1-macro, direction accuracy, binomial p-values,
    and Cohen's h effect size.
    Returns dict.
    """
    n = len(y_true)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    n_pred_classes = len(set(y_pred.tolist()))

    # Direction: classes 0-2 = down, 3-5 = up
    dir_true = (y_true >= 3).astype(int)
    dir_pred = (y_pred >= 3).astype(int)
    dir_acc  = (dir_true == dir_pred).mean()

    # Binomial test: direction vs 50 %
    n_correct_dir = int((dir_true == dir_pred).sum())
    p_dir = stats.binomtest(n_correct_dir, n, 0.5, 'greater').pvalue

    # Binomial test: accuracy vs majority baseline
    if majority_baseline_frac is None:
        majority_baseline_frac = Counter(y_true.tolist()).most_common(1)[0][1] / n
    n_correct_acc = int((y_true == y_pred).sum())
    p_maj = stats.binomtest(n_correct_acc, n, majority_baseline_frac, 'greater').pvalue

    # Cohen's h  (effect size for proportions)
    def _arcsin(p):
        return 2 * np.arcsin(np.sqrt(np.clip(p, 0, 1)))
    cohen_h_dir = _arcsin(dir_acc) - _arcsin(0.5)
    cohen_h_acc = _arcsin(acc) - _arcsin(majority_baseline_frac)

    return {
        'n': n,
        'acc': round(acc * 100, 2),
        'f1_macro': round(f1m * 100, 2),
        'dir_acc': round(dir_acc * 100, 2),
        'n_pred_classes': n_pred_classes,
        'pred_dist': dict(Counter(y_pred.tolist())),
        'majority_baseline': round(majority_baseline_frac * 100, 2),
        'dir_baseline': round(max(dir_true.mean(), 1 - dir_true.mean()) * 100, 2),
        'p_dir_vs_50': float(p_dir),
        'p_acc_vs_majority': float(p_maj),
        'cohen_h_dir': round(float(cohen_h_dir), 4),
        'cohen_h_acc': round(float(cohen_h_acc), 4),
    }


def significance_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


print('Model & evaluation functions ready')

In [ ]:
# Cell 5: Stage 2 -- Sector-Pooled Training Loop  (11 sectors)
# VGG16 via tf.data + XGBoost on VGG16 GAP features (512-dim, no re-read)

stage2_path = RESULTS_DIR / 'stage2_sector_results.json'

if stage2_path.exists():
    with open(stage2_path) as f:
        stage2_results = json.load(f)
else:
    stage2_results = {}

all_sectors = sorted(sector_tickers.keys())
done_sectors = set(stage2_results.keys())
todo_sectors = [s for s in all_sectors if s not in done_sectors]

print(f'Stage 2: Sector-Pooled Training (tf.data streaming)')
print(f'Total sectors: {len(all_sectors)}, Done: {len(done_sectors)}, '
      f'Remaining: {len(todo_sectors)}')
print('=' * 70)

t0_all = time.time()

for si, sector in enumerate(todo_sectors):
    t0 = time.time()
    n_stocks = len(sector_tickers[sector])
    print(f'\n[{si+1}/{len(todo_sectors)}] {sector}  ({n_stocks} stocks)')

    # ---- Build file lists (no images loaded yet) ----
    tickers = sector_tickers[sector]
    train_paths, train_labels, _ = _build_file_label_lists(tickers, 'train')
    test_paths, test_labels, test_tkrs = _build_file_label_lists(tickers, 'test')

    if len(train_paths) < 50 or len(test_paths) < 10:
        stage2_results[sector] = {'status': 'insufficient_data',
                                   'n_stocks': n_stocks,
                                   'n_train': len(train_paths),
                                   'n_test': len(test_paths)}
        print(f'  Skipped: train={len(train_paths)}, test={len(test_paths)}')
        continue

    print(f'  Files: train={len(train_paths):,}  test={len(test_paths):,}')

    y_train = np.array(train_labels, dtype=np.int32)
    y_test = np.array(test_labels, dtype=np.int32)

    result = {
        'sector': sector,
        'n_stocks': n_stocks,
        'n_train': len(train_paths),
        'n_test': len(test_paths),
        'train_class_dist': dict(Counter(y_train.tolist())),
        'test_class_dist': dict(Counter(y_test.tolist())),
        'status': 'ok',
    }

    # Majority baseline
    maj_cls, maj_cnt = Counter(y_test.tolist()).most_common(1)[0]
    maj_bl = maj_cnt / len(y_test)
    result['majority_baseline'] = round(maj_bl * 100, 2)
    dir_true = (y_test >= 3).astype(int)
    up_ratio = dir_true.mean()
    result['dir_baseline'] = round(max(up_ratio, 1 - up_ratio) * 100, 2)

    # Class weights
    cw = compute_class_weights_from_labels(y_train)

    # ---- Build tf.data datasets (shared for VGG16 + feature extraction) ----
    val_n = max(int(len(train_paths) * 0.15), 32)
    tr_paths, tr_labels = train_paths[:-val_n], train_labels[:-val_n]
    vl_paths, vl_labels = train_paths[-val_n:], train_labels[-val_n:]

    ds_tr = make_tf_dataset(tr_paths, tr_labels, batch_size=64, shuffle=True)
    ds_vl = make_tf_dataset(vl_paths, vl_labels, batch_size=64, shuffle=False)
    ds_te = make_tf_dataset(test_paths, test_labels, batch_size=64, shuffle=False)
    # For feature extraction (no shuffle, all train)
    ds_train_all = make_tf_dataset(train_paths, train_labels, batch_size=64, shuffle=False)

    # ---- VGG16 Training ----
    print('  Training VGG16_ft_block5...')
    try:
        tf.random.set_seed(42)
        np.random.seed(42)

        model = build_vgg16_ft_block5()
        history = model.fit(
            ds_tr, epochs=50, validation_data=ds_vl,
            class_weight=cw,
            callbacks=[
                EarlyStopping('val_loss', patience=5,
                              restore_best_weights=True, verbose=1),
                ReduceLROnPlateau('val_loss', factor=0.5, patience=2, verbose=1),
            ],
            verbose=1,
        )
        y_pred_vgg = np.argmax(model.predict(ds_te, verbose=0), axis=1)

        metrics_vgg = eval_predictions(y_test, y_pred_vgg, maj_bl)
        metrics_vgg['epochs_run'] = len(history.history['loss'])
        result['vgg16'] = metrics_vgg
        print(f'    VGG16: acc={metrics_vgg["acc"]}%, '
              f'dir={metrics_vgg["dir_acc"]}%, '
              f'f1m={metrics_vgg["f1_macro"]}%, '
              f'classes={metrics_vgg["n_pred_classes"]}/6')

        # ---- Extract VGG16 GAP features (512-dim) for XGBoost ----
        print('  Extracting VGG16 GAP features for XGBoost...')
        feat_extractor = build_vgg16_feature_extractor()  # frozen VGG16 → GAP → 512-dim

        feat_train = feat_extractor.predict(ds_train_all, verbose=0)
        feat_test = feat_extractor.predict(ds_te, verbose=0)
        print(f'    Features: train={feat_train.shape}, test={feat_test.shape}')

        del model, feat_extractor
        tf.keras.backend.clear_session(); gc.collect()
    except Exception as e:
        result['vgg16'] = {'status': f'error: {str(e)[:120]}'}
        print(f'    VGG16 ERROR: {str(e)[:120]}')
        tf.keras.backend.clear_session(); gc.collect()
        feat_train, feat_test = None, None

    # ---- XGBoost on VGG16 GAP features (512-dim, fast!) ----
    print('  Training XGBoost on GAP features...')
    try:
        if feat_train is not None and feat_test is not None:
            y_pred_xgb = train_xgboost_flat(feat_train, y_train,
                                             feat_test, y_test)
            if y_pred_xgb is not None:
                metrics_xgb = eval_predictions(y_test, y_pred_xgb, maj_bl)
                result['xgboost'] = metrics_xgb
                print(f'    XGB:   acc={metrics_xgb["acc"]}%, '
                      f'dir={metrics_xgb["dir_acc"]}%, '
                      f'f1m={metrics_xgb["f1_macro"]}%, '
                      f'classes={metrics_xgb["n_pred_classes"]}/6')
            else:
                result['xgboost'] = {'status': 'train_failed'}
            del feat_train, feat_test
            gc.collect()
        else:
            result['xgboost'] = {'status': 'skipped_no_features'}
    except Exception as e:
        result['xgboost'] = {'status': f'error: {str(e)[:120]}'}
        print(f'    XGB ERROR: {str(e)[:120]}')

    elapsed = time.time() - t0
    result['elapsed_sec'] = round(elapsed, 1)
    print(f'  Sector done in {elapsed:.0f}s')

    stage2_results[sector] = result
    gc.collect()

    # Checkpoint
    with open(stage2_path, 'w') as f:
        json.dump(stage2_results, f, indent=2)

total_time = time.time() - t0_all
print(f'\n{"=" * 70}')
print(f'Stage 2 complete: {len(stage2_results)} sectors  ({total_time:.0f}s)')

In [ ]:
# Cell 6: Stage 2 Results -- Sector Summary Table + Binomial Tests + Bonferroni

# Reload (in case of restart)
with open(stage2_path) as f:
    stage2_results = json.load(f)

# ------------------------------------------------------------------
# Build summary DataFrame
# ------------------------------------------------------------------
rows = []
for sector in sorted(stage2_results.keys()):
    r = stage2_results[sector]
    if r.get('status') != 'ok':
        continue
    row = {
        'Sector':        sector,
        'Stocks':        r['n_stocks'],
        'Train':         r['n_train'],
        'Test':          r['n_test'],
        'Maj BL (%)':    r['majority_baseline'],
        'Dir BL (%)':    r['dir_baseline'],
    }
    for tag, key in [('VGG', 'vgg16'), ('XGB', 'xgboost')]:
        m = r.get(key, {})
        if 'acc' in m:
            row[f'{tag} Acc']   = m['acc']
            row[f'{tag} F1m']   = m['f1_macro']
            row[f'{tag} Dir']   = m['dir_acc']
            row[f'{tag} #Cls']  = m['n_pred_classes']
            row[f'{tag} p_dir'] = m['p_dir_vs_50']
            row[f'{tag} p_maj'] = m['p_acc_vs_majority']
            row[f'{tag} h_dir'] = m['cohen_h_dir']
            row[f'{tag} h_acc'] = m['cohen_h_acc']
        else:
            for c in ['Acc','F1m','Dir','#Cls','p_dir','p_maj','h_dir','h_acc']:
                row[f'{tag} {c}'] = None
    rows.append(row)

df_s2 = pd.DataFrame(rows)

# ------------------------------------------------------------------
# Multiple testing correction (Bonferroni & FDR)
# ------------------------------------------------------------------
n_sectors = len(df_s2)

for tag in ['VGG', 'XGB']:
    for test_col, suffix in [('p_dir', 'dir'), ('p_maj', 'maj')]:
        raw_p = df_s2[f'{tag} {test_col}'].values.astype(float)
        # Bonferroni
        bonf = np.minimum(raw_p * n_sectors, 1.0)
        df_s2[f'{tag} bonf_{suffix}'] = np.round(bonf, 6)
        # BH-FDR
        reject, fdr_p, _, _ = multipletests(raw_p, method='fdr_bh')
        df_s2[f'{tag} fdr_{suffix}'] = np.round(fdr_p, 6)
        df_s2[f'{tag} sig_{suffix}'] = [significance_stars(p) for p in bonf]

# ------------------------------------------------------------------
# Print compact table
# ------------------------------------------------------------------
display_cols = ['Sector', 'Stocks', 'Train', 'Test', 'Maj BL (%)', 'Dir BL (%)',
                'VGG Acc', 'VGG Dir', 'VGG F1m', 'VGG #Cls',
                'VGG sig_dir', 'VGG sig_maj',
                'XGB Acc', 'XGB Dir', 'XGB F1m', 'XGB #Cls',
                'XGB sig_dir', 'XGB sig_maj']
print('\n=== Stage 2: Sector-Pooled Results (Bonferroni-corrected significance) ===')
print(df_s2[display_cols].to_string(index=False))

# ------------------------------------------------------------------
# Detailed p-value table
# ------------------------------------------------------------------
p_cols = ['Sector']
for tag in ['VGG', 'XGB']:
    for test in ['dir', 'maj']:
        p_cols += [f'{tag} p_{test}', f'{tag} bonf_{test}',
                   f'{tag} fdr_{test}', f'{tag} h_{test}']
print('\n=== P-values, Bonferroni, FDR, Cohen\'s h ===')
print(df_s2[p_cols].to_string(index=False))

# Save
df_s2.to_csv(RESULTS_DIR / 'stage2_sector_summary.csv', index=False)
print(f'\nSaved to {RESULTS_DIR / "stage2_sector_summary.csv"}')

In [ ]:
# Cell 7: Stage 2.5 -- Financial Category Grouping + Correlation Analysis

# ------------------------------------------------------------------
# 7-A  Aggregate by financial category
# ------------------------------------------------------------------
df_s2['Category'] = df_s2['Sector'].map(SECTOR_TO_CATEGORY)

cat_rows = []
for cat in ['Cyclical Growth', 'Cyclical Value', 'Defensive']:
    sub = df_s2[df_s2['Category'] == cat]
    row = {'Category': cat, 'N_sectors': len(sub),
           'N_stocks': sub['Stocks'].sum(),
           'N_train': sub['Train'].sum(),
           'N_test': sub['Test'].sum()}
    for tag in ['VGG', 'XGB']:
        for m in ['Acc', 'Dir', 'F1m']:
            vals = sub[f'{tag} {m}'].dropna()
            row[f'{tag} {m} mean'] = round(vals.mean(), 2) if len(vals) else None
            row[f'{tag} {m} std']  = round(vals.std(), 2)  if len(vals) else None
        # Count significant sectors (Bonferroni)
        row[f'{tag} sig_dir_count'] = int(
            (sub[f'{tag} bonf_dir'] < 0.05).sum()) if f'{tag} bonf_dir' in sub else 0
        row[f'{tag} sig_maj_count'] = int(
            (sub[f'{tag} bonf_maj'] < 0.05).sum()) if f'{tag} bonf_maj' in sub else 0
    cat_rows.append(row)

df_cat = pd.DataFrame(cat_rows)
print('=== Stage 2.5: Financial Category Summary ===')
print(df_cat.to_string(index=False))

# ------------------------------------------------------------------
# 7-B  Compute sector-level volatility & volume from price data
# ------------------------------------------------------------------
print('\nComputing sector volatility & volume from price data...')

sector_vol_info = {}
for sector in all_sectors:
    vols, avg_volumes = [], []
    for ticker in sector_tickers[sector]:
        pkl_path = PRICE_DIR / f'{ticker}.pkl'
        if not pkl_path.exists():
            continue
        try:
            df_price = pd.read_pickle(pkl_path)
            # Use test period for metrics
            df_test = df_price[df_price.index >= TEST_START]
            if len(df_test) < 20:
                continue
            daily_ret = df_test['Close'].pct_change().dropna()
            vols.append(daily_ret.std() * np.sqrt(252) * 100)  # annualised vol %
            avg_volumes.append(df_test['Volume'].mean())
        except Exception:
            continue
    sector_vol_info[sector] = {
        'mean_volatility': round(np.mean(vols), 2) if vols else None,
        'median_volatility': round(np.median(vols), 2) if vols else None,
        'mean_volume': round(np.mean(avg_volumes), 0) if avg_volumes else None,
    }

df_vol = pd.DataFrame(sector_vol_info).T.reset_index().rename(
    columns={'index': 'Sector'})

# Merge with performance
df_corr = df_s2[['Sector', 'VGG Acc', 'VGG Dir', 'XGB Acc', 'XGB Dir']].merge(
    df_vol, on='Sector', how='inner')

print('\n=== Sector Volatility & Performance ===')
print(df_corr.to_string(index=False))

# ------------------------------------------------------------------
# 7-C  Correlation: volatility vs model performance
# ------------------------------------------------------------------
print('\n=== Correlation Analysis: Volatility vs Model Performance ===')
for model_tag in ['VGG', 'XGB']:
    for metric in ['Acc', 'Dir']:
        col = f'{model_tag} {metric}'
        valid = df_corr[['mean_volatility', col]].dropna()
        if len(valid) < 4:
            continue
        r_pearson, p_pearson = stats.pearsonr(valid['mean_volatility'], valid[col])
        r_spearman, p_spearman = stats.spearmanr(valid['mean_volatility'], valid[col])
        print(f'  {model_tag} {metric} vs Volatility:  '
              f'Pearson r={r_pearson:.3f} (p={p_pearson:.4f})  '
              f'Spearman rho={r_spearman:.3f} (p={p_spearman:.4f})')

# Save
df_cat.to_csv(RESULTS_DIR / 'stage2_5_category_summary.csv', index=False)
df_corr.to_csv(RESULTS_DIR / 'stage2_5_volatility_performance.csv', index=False)
with open(RESULTS_DIR / 'stage2_5_sector_volatility.json', 'w') as f:
    json.dump(sector_vol_info, f, indent=2)
print('\nStage 2.5 results saved.')

In [ ]:
# Cell 8: Stage 2.5 Visualizations

fig = plt.figure(figsize=(22, 18))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.30)

# Sort sectors by VGG Dir accuracy for consistent ordering
df_plot = df_s2.sort_values('VGG Dir', ascending=True).reset_index(drop=True)

# ------------------------------------------------------------------
# (a) Sector accuracy  (VGG & XGB side-by-side)
# ------------------------------------------------------------------
ax = fig.add_subplot(gs[0, 0])
y = np.arange(len(df_plot))
h = 0.35
ax.barh(y - h/2, df_plot['VGG Acc'], height=h, label='VGG16', color='steelblue', alpha=0.85)
ax.barh(y + h/2, df_plot['XGB Acc'], height=h, label='XGBoost', color='coral', alpha=0.85)
ax.scatter(df_plot['Maj BL (%)'], y, color='black', marker='|', s=200,
           zorder=5, label='Majority BL')
ax.set_yticks(y); ax.set_yticklabels(df_plot['Sector'], fontsize=9)
ax.set_xlabel('Accuracy (%)')
ax.set_title('(a) 6-Class Accuracy by Sector')
ax.legend(fontsize=8, loc='lower right')

# ------------------------------------------------------------------
# (b) Direction accuracy
# ------------------------------------------------------------------
ax = fig.add_subplot(gs[0, 1])
ax.barh(y - h/2, df_plot['VGG Dir'], height=h, label='VGG16', color='steelblue', alpha=0.85)
ax.barh(y + h/2, df_plot['XGB Dir'], height=h, label='XGBoost', color='coral', alpha=0.85)
ax.axvline(50, color='red', ls='--', lw=1, label='50% (chance)')
ax.scatter(df_plot['Dir BL (%)'], y, color='black', marker='|', s=200,
           zorder=5, label='Dir Majority BL')
ax.set_yticks(y); ax.set_yticklabels(df_plot['Sector'], fontsize=9)
ax.set_xlabel('Direction Accuracy (%)')
ax.set_title('(b) Direction Accuracy by Sector')
ax.legend(fontsize=8, loc='lower right')

# ------------------------------------------------------------------
# (c) Cohen's h heatmap
# ------------------------------------------------------------------
ax = fig.add_subplot(gs[1, 0])
heat_data = df_plot[['Sector', 'VGG h_dir', 'VGG h_acc',
                      'XGB h_dir', 'XGB h_acc']].set_index('Sector')
heat_data.columns = ['VGG Dir h', 'VGG Acc h', 'XGB Dir h', 'XGB Acc h']
sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5)
ax.set_title("(c) Cohen's h Effect Size (green=positive)")

# ------------------------------------------------------------------
# (d) Category comparison  (grouped bar)
# ------------------------------------------------------------------
ax = fig.add_subplot(gs[1, 1])
cats = ['Cyclical Growth', 'Cyclical Value', 'Defensive']
x_cat = np.arange(len(cats))
w = 0.18
for i, (metric, color) in enumerate([
    ('VGG Dir mean', 'steelblue'), ('VGG Acc mean', 'royalblue'),
    ('XGB Dir mean', 'coral'),     ('XGB Acc mean', 'orangered')]):
    vals = [df_cat.loc[df_cat['Category'] == c, metric].values[0]
            for c in cats]
    ax.bar(x_cat + (i - 1.5) * w, vals, width=w, label=metric, color=color,
           alpha=0.85)
ax.axhline(50, color='grey', ls=':', lw=1)
ax.set_xticks(x_cat); ax.set_xticklabels(cats, fontsize=10)
ax.set_ylabel('Accuracy / Direction (%)')
ax.set_title('(d) Performance by Financial Category')
ax.legend(fontsize=7, ncol=2)

# ------------------------------------------------------------------
# (e) Volatility vs VGG Dir accuracy scatter
# ------------------------------------------------------------------
ax = fig.add_subplot(gs[2, 0])
valid = df_corr.dropna(subset=['mean_volatility', 'VGG Dir'])
colors_cat = {'Cyclical Growth': 'steelblue', 'Cyclical Value': 'coral',
              'Defensive': 'green'}
for _, row_data in valid.iterrows():
    cat = SECTOR_TO_CATEGORY.get(row_data['Sector'], 'Other')
    ax.scatter(row_data['mean_volatility'], row_data['VGG Dir'],
               color=colors_cat.get(cat, 'grey'), s=80, zorder=5)
    ax.annotate(row_data['Sector'][:12], (row_data['mean_volatility'],
                row_data['VGG Dir']), fontsize=7, ha='left', va='bottom')
# Trend line
if len(valid) >= 4:
    z = np.polyfit(valid['mean_volatility'], valid['VGG Dir'], 1)
    xr = np.linspace(valid['mean_volatility'].min(), valid['mean_volatility'].max(), 50)
    ax.plot(xr, np.polyval(z, xr), 'k--', alpha=0.5)
    rho, p_rho = stats.spearmanr(valid['mean_volatility'], valid['VGG Dir'])
    ax.set_title(f'(e) Volatility vs VGG Dir Acc  (rho={rho:.2f}, p={p_rho:.3f})')
ax.axhline(50, color='red', ls=':', lw=1)
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('VGG Direction Accuracy (%)')
# Legend for categories
for cat, col in colors_cat.items():
    ax.scatter([], [], color=col, label=cat, s=60)
ax.legend(fontsize=7)

# ------------------------------------------------------------------
# (f) Volatility vs XGB Dir accuracy scatter
# ------------------------------------------------------------------
ax = fig.add_subplot(gs[2, 1])
valid2 = df_corr.dropna(subset=['mean_volatility', 'XGB Dir'])
for _, row_data in valid2.iterrows():
    cat = SECTOR_TO_CATEGORY.get(row_data['Sector'], 'Other')
    ax.scatter(row_data['mean_volatility'], row_data['XGB Dir'],
               color=colors_cat.get(cat, 'grey'), s=80, zorder=5)
    ax.annotate(row_data['Sector'][:12], (row_data['mean_volatility'],
                row_data['XGB Dir']), fontsize=7, ha='left', va='bottom')
if len(valid2) >= 4:
    z = np.polyfit(valid2['mean_volatility'], valid2['XGB Dir'], 1)
    xr = np.linspace(valid2['mean_volatility'].min(), valid2['mean_volatility'].max(), 50)
    ax.plot(xr, np.polyval(z, xr), 'k--', alpha=0.5)
    rho2, p_rho2 = stats.spearmanr(valid2['mean_volatility'], valid2['XGB Dir'])
    ax.set_title(f'(f) Volatility vs XGB Dir Acc  (rho={rho2:.2f}, p={p_rho2:.3f})')
ax.axhline(50, color='red', ls=':', lw=1)
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('XGB Direction Accuracy (%)')
for cat, col in colors_cat.items():
    ax.scatter([], [], color=col, label=cat, s=60)
ax.legend(fontsize=7)

plt.savefig(str(RESULTS_DIR / 'stage2_5_visualizations.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Stage 2.5 visualizations saved.')

In [ ]:
# Cell 9: Stage 3 -- Full Pooled VGG16 Training  (tf.data pipeline)
#
# All 1.3M+ images combined into a single model.
# Uses tf.data to stream from disk -- no OOM.

print('=' * 70)
print('Stage 3: Full Pooled Training -- VGG16')
print('=' * 70)

# ---- Build file lists ----
print('Building file lists for all stocks...')
t0 = time.time()

train_paths, train_labels, train_tkrs = _build_file_label_lists(ok_tickers, 'train')
test_paths,  test_labels,  test_tkrs  = _build_file_label_lists(ok_tickers, 'test')

train_labels_arr = np.array(train_labels, dtype=np.int32)
test_labels_arr  = np.array(test_labels,  dtype=np.int32)
test_tkrs_arr    = np.array(test_tkrs)

print(f'  Train files: {len(train_paths):,}')
print(f'  Test files:  {len(test_paths):,}')
print(f'  Train class dist: {dict(sorted(Counter(train_labels).items()))}')
print(f'  Test class dist:  {dict(sorted(Counter(test_labels).items()))}')
print(f'  File list built in {time.time()-t0:.0f}s')

# Baselines
pool_maj_cls, pool_maj_cnt = Counter(test_labels).most_common(1)[0]
pool_maj_bl = pool_maj_cnt / len(test_labels)
pool_dir_true = (test_labels_arr >= 3).astype(int)
pool_up_ratio = pool_dir_true.mean()
pool_dir_bl = max(pool_up_ratio, 1 - pool_up_ratio)
print(f'  Majority baseline: {pool_maj_bl*100:.2f}%  Dir baseline: {pool_dir_bl*100:.2f}%')

# ---- Class weights ----
cw_pool = compute_class_weights_from_labels(train_labels_arr)
print(f'  Class weights: { {k: round(v,3) for k,v in cw_pool.items()} }')

# ---- Validation split: last 15% of training paths ----
val_n = int(len(train_paths) * 0.15)
val_paths,  val_labels  = train_paths[-val_n:],  train_labels[-val_n:]
tr_paths,   tr_labels   = train_paths[:-val_n],  train_labels[:-val_n]
print(f'  After val split: train={len(tr_paths):,}, val={len(val_paths):,}')

# ---- Build tf.data datasets ----
BATCH_SIZE = 128

ds_train = make_tf_dataset(tr_paths,   tr_labels,   BATCH_SIZE, shuffle=True)
ds_val   = make_tf_dataset(val_paths,  val_labels,  BATCH_SIZE, shuffle=False)
ds_test  = make_tf_dataset(test_paths, test_labels, BATCH_SIZE, shuffle=False)

# ---- Train ----
print('\nTraining VGG16_ft_block5 on full pooled data...')
t0 = time.time()
model_pooled_vgg, hist_vgg = train_vgg16_on_tfdata(
    ds_train, ds_val, cw_pool, epochs=50, patience=5)
vgg_train_time = time.time() - t0
print(f'Training completed in {vgg_train_time:.0f}s  '
      f'({len(hist_vgg.history["loss"])} epochs)')

# ---- Predict ----
print('Predicting on test set...')
y_proba_pool_vgg = model_pooled_vgg.predict(ds_test, verbose=1)
y_pred_pool_vgg  = np.argmax(y_proba_pool_vgg, axis=1)

# ---- Evaluate ----
pool_vgg_metrics = eval_predictions(test_labels_arr, y_pred_pool_vgg, pool_maj_bl)
pool_vgg_metrics['train_time_sec'] = round(vgg_train_time, 1)
pool_vgg_metrics['epochs_run'] = len(hist_vgg.history['loss'])

print(f'\n=== Pooled VGG16 Results ===')
print(f'  Acc:  {pool_vgg_metrics["acc"]}%  (majority BL: {pool_maj_bl*100:.2f}%) '
      f'{significance_stars(pool_vgg_metrics["p_acc_vs_majority"])}')
print(f'  Dir:  {pool_vgg_metrics["dir_acc"]}%  (dir BL: {pool_dir_bl*100:.2f}%) '
      f'{significance_stars(pool_vgg_metrics["p_dir_vs_50"])}')
print(f'  F1m:  {pool_vgg_metrics["f1_macro"]}%')
print(f'  Classes predicted: {pool_vgg_metrics["n_pred_classes"]}/6')
print(f'  Pred dist: {pool_vgg_metrics["pred_dist"]}')
print(f'  Cohen\'s h (dir): {pool_vgg_metrics["cohen_h_dir"]}')
print(f'  Mode collapse: '
      f'{"YES" if pool_vgg_metrics["n_pred_classes"] <= 2 else "NO"}')

# Clean up model to free GPU memory
del model_pooled_vgg
tf.keras.backend.clear_session()
gc.collect()

# Checkpoint
with open(RESULTS_DIR / 'stage3_vgg_results.json', 'w') as f:
    json.dump(pool_vgg_metrics, f, indent=2)
np.save(str(RESULTS_DIR / 'stage3_vgg_y_pred.npy'), y_pred_pool_vgg)
print('\nStage 3 VGG results checkpointed to Drive.')

In [ ]:
# Cell 10: Stage 3 -- Full Pooled XGBoost  (64x64 flatten via tf.data)

print('=' * 70)
print('Stage 3: Full Pooled Training -- XGBoost (64x64 flatten)')
print('=' * 70)

# ---- Collect features via tf.data (batched, avoids OOM) ----
print('Extracting 64x64 features from disk via tf.data...')
t0 = time.time()

ds_train_xgb = make_tf_dataset(train_paths, train_labels, 512,
                                shuffle=False, parse_fn=_parse_image_small)
ds_test_xgb  = make_tf_dataset(test_paths,  test_labels,  512,
                                shuffle=False, parse_fn=_parse_image_small)

# Collect into numpy (streaming, not all at once if too large)
# For XGBoost we need full arrays. With 1.3M * 12288 floats ~ 60 GB.
# This is too large -- subsample training to a manageable size.

MAX_XGB_TRAIN = 300_000  # Subsample training if > 300K

if len(train_paths) > MAX_XGB_TRAIN:
    print(f'  Subsampling training from {len(train_paths):,} to {MAX_XGB_TRAIN:,}')
    rng = np.random.RandomState(42)
    idx_sub = rng.choice(len(train_paths), MAX_XGB_TRAIN, replace=False)
    idx_sub.sort()
    xgb_train_paths  = [train_paths[i] for i in idx_sub]
    xgb_train_labels = [train_labels[i] for i in idx_sub]
else:
    xgb_train_paths  = train_paths
    xgb_train_labels = train_labels

# Build datasets for the subsampled set
ds_xgb_tr = make_tf_dataset(xgb_train_paths, xgb_train_labels, 512,
                             shuffle=False, parse_fn=_parse_image_small)
ds_xgb_te = make_tf_dataset(test_paths, test_labels, 512,
                             shuffle=False, parse_fn=_parse_image_small)

# Collect to numpy
print('  Collecting training features...')
X_tr_chunks, y_tr_chunks = [], []
for xb, yb in ds_xgb_tr:
    X_tr_chunks.append(xb.numpy())
    y_tr_chunks.append(yb.numpy())
X_pool_xgb_train = np.concatenate(X_tr_chunks)
y_pool_xgb_train = np.concatenate(y_tr_chunks)
del X_tr_chunks, y_tr_chunks; gc.collect()

print('  Collecting test features...')
X_te_chunks, y_te_chunks = [], []
for xb, yb in ds_xgb_te:
    X_te_chunks.append(xb.numpy())
    y_te_chunks.append(yb.numpy())
X_pool_xgb_test = np.concatenate(X_te_chunks)
y_pool_xgb_test = np.concatenate(y_te_chunks)
del X_te_chunks, y_te_chunks; gc.collect()

print(f'  Train: {X_pool_xgb_train.shape}  Test: {X_pool_xgb_test.shape}')
print(f'  Feature extraction took {time.time()-t0:.0f}s')

# ---- Train XGBoost ----
print('\nTraining XGBoost...')
t0 = time.time()

cw_xgb = compute_class_weights_from_labels(y_pool_xgb_train)
sw_xgb = np.array([cw_xgb[y] for y in y_pool_xgb_train])

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.01,
    objective='multi:softmax', num_class=NUM_CLASSES,
    random_state=42, use_label_encoder=False,
    eval_metric='mlogloss', tree_method='hist',
    n_jobs=-1,
)
xgb_model.fit(X_pool_xgb_train, y_pool_xgb_train,
              sample_weight=sw_xgb, verbose=True)
y_pred_pool_xgb = xgb_model.predict(X_pool_xgb_test)
xgb_train_time = time.time() - t0

# Evaluate
pool_xgb_metrics = eval_predictions(test_labels_arr, y_pred_pool_xgb, pool_maj_bl)
pool_xgb_metrics['train_time_sec'] = round(xgb_train_time, 1)
pool_xgb_metrics['n_train_subsampled'] = len(y_pool_xgb_train)

print(f'\n=== Pooled XGBoost Results ===')
print(f'  Acc:  {pool_xgb_metrics["acc"]}%  (majority BL: {pool_maj_bl*100:.2f}%) '
      f'{significance_stars(pool_xgb_metrics["p_acc_vs_majority"])}')
print(f'  Dir:  {pool_xgb_metrics["dir_acc"]}%  (dir BL: {pool_dir_bl*100:.2f}%) '
      f'{significance_stars(pool_xgb_metrics["p_dir_vs_50"])}')
print(f'  F1m:  {pool_xgb_metrics["f1_macro"]}%')
print(f'  Classes predicted: {pool_xgb_metrics["n_pred_classes"]}/6')
print(f'  Mode collapse: '
      f'{"YES" if pool_xgb_metrics["n_pred_classes"] <= 2 else "NO"}')

# Clean up
del xgb_model, X_pool_xgb_train, y_pool_xgb_train
del X_pool_xgb_test, y_pool_xgb_test
gc.collect()

# Checkpoint
with open(RESULTS_DIR / 'stage3_xgb_results.json', 'w') as f:
    json.dump(pool_xgb_metrics, f, indent=2)
np.save(str(RESULTS_DIR / 'stage3_xgb_y_pred.npy'), y_pred_pool_xgb)
print('Stage 3 XGBoost results checkpointed to Drive.')

In [ ]:
# Cell 11: Stage 3 -- Permutation Test  (shuffled labels null baseline)
#
# Train the SAME VGG16 architecture on randomly shuffled labels.
# If real accuracy >> permutation accuracy, signal is genuine.

print('=' * 70)
print('Stage 3: Permutation Test (shuffled labels)')
print('=' * 70)

# Shuffle training labels (keep test labels real for evaluation)
rng_perm = np.random.RandomState(99)
train_labels_shuffled = rng_perm.permutation(train_labels).tolist()

print(f'Original train label dist: {dict(sorted(Counter(train_labels).items()))}')
print(f'Shuffled train label dist: {dict(sorted(Counter(train_labels_shuffled).items()))}')
print('(distributions match, only assignment is randomised)\n')

# Validation split
shuf_val_labels  = train_labels_shuffled[-val_n:]
shuf_tr_labels   = train_labels_shuffled[:-val_n]

ds_shuf_train = make_tf_dataset(tr_paths,  shuf_tr_labels,  BATCH_SIZE, shuffle=True)
ds_shuf_val   = make_tf_dataset(val_paths, shuf_val_labels, BATCH_SIZE, shuffle=False)

cw_shuf = compute_class_weights_from_labels(train_labels_shuffled)

print('Training VGG16 on shuffled labels...')
t0 = time.time()
model_perm, hist_perm = train_vgg16_on_tfdata(
    ds_shuf_train, ds_shuf_val, cw_shuf, epochs=30, patience=3)
perm_time = time.time() - t0
print(f'Permutation training done in {perm_time:.0f}s  '
      f'({len(hist_perm.history["loss"])} epochs)')

# Predict on REAL test labels
y_proba_perm = model_perm.predict(ds_test, verbose=0)
y_pred_perm  = np.argmax(y_proba_perm, axis=1)

perm_metrics = eval_predictions(test_labels_arr, y_pred_perm, pool_maj_bl)

print(f'\n=== Permutation Test Results ===')
print(f'  Real VGG Acc:  {pool_vgg_metrics["acc"]}%')
print(f'  Perm VGG Acc:  {perm_metrics["acc"]}%')
print(f'  Real VGG Dir:  {pool_vgg_metrics["dir_acc"]}%')
print(f'  Perm VGG Dir:  {perm_metrics["dir_acc"]}%')
print(f'  Acc gap (real - perm): '
      f'{pool_vgg_metrics["acc"] - perm_metrics["acc"]:.2f} pp')
print(f'  Dir gap (real - perm): '
      f'{pool_vgg_metrics["dir_acc"] - perm_metrics["dir_acc"]:.2f} pp')

# One-sided test: is real accuracy significantly higher than permutation?
# Use binomial: real correct count vs permutation accuracy as null rate
perm_rate = perm_metrics['acc'] / 100
real_correct = int(accuracy_score(test_labels_arr, y_pred_pool_vgg) * len(test_labels_arr))
p_perm = stats.binomtest(real_correct, len(test_labels_arr),
                          perm_rate, 'greater').pvalue
print(f'  Binomial test (real vs perm rate): p={p_perm:.6e} '
      f'{significance_stars(p_perm)}')

signal_genuine = pool_vgg_metrics['acc'] > perm_metrics['acc'] and p_perm < 0.05
print(f'  Signal genuine: {"YES" if signal_genuine else "NO"}')

del model_perm
tf.keras.backend.clear_session()
gc.collect()

# Save
perm_result = {
    'real_vgg': pool_vgg_metrics,
    'perm_vgg': perm_metrics,
    'acc_gap_pp': round(pool_vgg_metrics['acc'] - perm_metrics['acc'], 2),
    'dir_gap_pp': round(pool_vgg_metrics['dir_acc'] - perm_metrics['dir_acc'], 2),
    'p_real_vs_perm': float(p_perm),
    'signal_genuine': signal_genuine,
}
with open(RESULTS_DIR / 'stage3_permutation_test.json', 'w') as f:
    json.dump(perm_result, f, indent=2)
print('Permutation test results saved.')

In [ ]:
# Cell 12: Stage 3 -- Per-Sector Decomposition + Comparison with Stage 2

print('=' * 70)
print('Stage 3: Per-Sector Decomposition of Pooled Model')
print('=' * 70)

# ---- Per-sector breakdown of pooled predictions ----
pooled_sector_rows = []

for sector in sorted(all_sectors):
    mask = np.array([ticker_sector.get(t, '') == sector for t in test_tkrs_arr])
    if mask.sum() < 10:
        continue

    y_true_s = test_labels_arr[mask]
    y_vgg_s  = y_pred_pool_vgg[mask]
    y_xgb_s  = y_pred_pool_xgb[mask]

    maj_s = Counter(y_true_s.tolist()).most_common(1)[0][1] / len(y_true_s)

    row = {'Sector': sector, 'N_test': int(mask.sum())}

    for tag, yp in [('Pool_VGG', y_vgg_s), ('Pool_XGB', y_xgb_s)]:
        m = eval_predictions(y_true_s, yp, maj_s)
        row[f'{tag} Acc']   = m['acc']
        row[f'{tag} Dir']   = m['dir_acc']
        row[f'{tag} F1m']   = m['f1_macro']
        row[f'{tag} #Cls']  = m['n_pred_classes']
        row[f'{tag} h_dir'] = m['cohen_h_dir']

    # Stage 2 specialist results for comparison
    s2 = stage2_results.get(sector, {})
    for tag_s2, key_s2 in [('S2_VGG', 'vgg16'), ('S2_XGB', 'xgboost')]:
        m2 = s2.get(key_s2, {})
        row[f'{tag_s2} Acc'] = m2.get('acc')
        row[f'{tag_s2} Dir'] = m2.get('dir_acc')
        row[f'{tag_s2} F1m'] = m2.get('f1_macro')

    pooled_sector_rows.append(row)

df_decomp = pd.DataFrame(pooled_sector_rows)

# Compute deltas: Pooled - Specialist
df_decomp['VGG Acc delta'] = (df_decomp['Pool_VGG Acc'].astype(float) -
                               df_decomp['S2_VGG Acc'].astype(float)).round(2)
df_decomp['XGB Acc delta'] = (df_decomp['Pool_XGB Acc'].astype(float) -
                               df_decomp['S2_XGB Acc'].astype(float)).round(2)
df_decomp['VGG Dir delta'] = (df_decomp['Pool_VGG Dir'].astype(float) -
                               df_decomp['S2_VGG Dir'].astype(float)).round(2)
df_decomp['XGB Dir delta'] = (df_decomp['Pool_XGB Dir'].astype(float) -
                               df_decomp['S2_XGB Dir'].astype(float)).round(2)

print('\n=== Specialist (Stage 2) vs Generalist (Stage 3) ===')
comp_cols = ['Sector', 'N_test',
             'S2_VGG Acc', 'Pool_VGG Acc', 'VGG Acc delta',
             'S2_VGG Dir', 'Pool_VGG Dir', 'VGG Dir delta',
             'S2_XGB Acc', 'Pool_XGB Acc', 'XGB Acc delta',
             'S2_XGB Dir', 'Pool_XGB Dir', 'XGB Dir delta']
print(df_decomp[comp_cols].to_string(index=False))

# Summary statistics
print(f'\n--- Summary ---')
for tag in ['VGG', 'XGB']:
    acc_d = df_decomp[f'{tag} Acc delta'].dropna()
    dir_d = df_decomp[f'{tag} Dir delta'].dropna()
    if len(acc_d) > 0:
        print(f'{tag} Acc delta:  mean={acc_d.mean():.2f} pp, '
              f'median={acc_d.median():.2f} pp  '
              f'(pooled {"better" if acc_d.mean()>0 else "worse"} on average)')
    if len(dir_d) > 0:
        print(f'{tag} Dir delta:  mean={dir_d.mean():.2f} pp, '
              f'median={dir_d.median():.2f} pp  '
              f'(pooled {"better" if dir_d.mean()>0 else "worse"} on average)')

# ---- Visualization: decomposition comparison ----
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

df_d = df_decomp.sort_values('VGG Dir delta', ascending=True).reset_index(drop=True)
y_pos = np.arange(len(df_d))

# VGG Dir: specialist vs generalist
ax = axes[0]
ax.barh(y_pos - 0.2, df_d['S2_VGG Dir'], height=0.35,
        label='Specialist (S2)', color='steelblue', alpha=0.8)
ax.barh(y_pos + 0.2, df_d['Pool_VGG Dir'], height=0.35,
        label='Generalist (S3)', color='coral', alpha=0.8)
ax.axvline(50, color='red', ls=':', lw=1)
ax.set_yticks(y_pos); ax.set_yticklabels(df_d['Sector'], fontsize=9)
ax.set_xlabel('Direction Accuracy (%)')
ax.set_title('VGG16: Specialist vs Generalist')
ax.legend(fontsize=9)

# XGB Dir: specialist vs generalist
ax = axes[1]
ax.barh(y_pos - 0.2, df_d['S2_XGB Dir'], height=0.35,
        label='Specialist (S2)', color='steelblue', alpha=0.8)
ax.barh(y_pos + 0.2, df_d['Pool_XGB Dir'], height=0.35,
        label='Generalist (S3)', color='coral', alpha=0.8)
ax.axvline(50, color='red', ls=':', lw=1)
ax.set_yticks(y_pos); ax.set_yticklabels(df_d['Sector'], fontsize=9)
ax.set_xlabel('Direction Accuracy (%)')
ax.set_title('XGBoost: Specialist vs Generalist')
ax.legend(fontsize=9)

plt.suptitle('Stage 2 (Sector Specialist) vs Stage 3 (Full Pooled Generalist)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'stage3_specialist_vs_generalist.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ---- Confusion matrices for pooled model ----
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for idx, (name, yp) in enumerate([('Pooled VGG16', y_pred_pool_vgg),
                                    ('Pooled XGBoost', y_pred_pool_xgb)]):
    cm = confusion_matrix(test_labels_arr, yp, labels=list(range(6)))
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    ax = axes[idx]
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES_6, yticklabels=CLASS_NAMES_6)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    m = eval_predictions(test_labels_arr, yp, pool_maj_bl)
    ax.set_title(f'{name}\nAcc={m["acc"]}% Dir={m["dir_acc"]}% '
                 f'F1m={m["f1_macro"]}% Cls={m["n_pred_classes"]}/6')
plt.suptitle(f'Pooled Model Confusion Matrices  '
             f'({len(train_paths):,} train images)', fontsize=13)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'stage3_confusion_matrices.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Save decomposition
df_decomp.to_csv(RESULTS_DIR / 'stage3_sector_decomposition.csv', index=False)
print('Stage 3 decomposition saved.')

In [ ]:
# Cell 13: Final Summary -- All results consolidated & saved to Drive

print('=' * 70)
print('  FINAL SUMMARY: Cross-Sectional Sector Analysis')
print('=' * 70)

# ---- Stage 2 summary ----
n_s2 = len([s for s in stage2_results.values() if s.get('status') == 'ok'])
print(f'\n1. STAGE 2 -- Sector-Pooled Training ({n_s2} sectors)')

for tag, key in [('VGG16', 'vgg16'), ('XGBoost', 'xgboost')]:
    accs, dirs = [], []
    sig_dir, sig_maj = 0, 0
    for sec, r in stage2_results.items():
        m = r.get(key, {})
        if 'acc' in m:
            accs.append(m['acc'])
            dirs.append(m['dir_acc'])
    # Use Bonferroni-corrected from df_s2
    tag_short = 'VGG' if tag == 'VGG16' else 'XGB'
    sig_dir = int((df_s2[f'{tag_short} bonf_dir'] < 0.05).sum())
    sig_maj = int((df_s2[f'{tag_short} bonf_maj'] < 0.05).sum())
    print(f'  {tag}:')
    print(f'    Acc:  mean={np.mean(accs):.1f}%  (range {min(accs):.1f}-{max(accs):.1f}%)')
    print(f'    Dir:  mean={np.mean(dirs):.1f}%  (range {min(dirs):.1f}-{max(dirs):.1f}%)')
    print(f'    Sig (Bonferroni): {sig_dir}/{n_s2} sectors Dir>50%, '
          f'{sig_maj}/{n_s2} sectors Acc>Majority')

# ---- Stage 2.5 summary ----
print(f'\n2. STAGE 2.5 -- Financial Category Grouping')
for _, crow in df_cat.iterrows():
    print(f'  {crow["Category"]:>16s}:  '
          f'VGG Dir={crow["VGG Dir mean"]:.1f}%  '
          f'XGB Dir={crow["XGB Dir mean"]:.1f}%  '
          f'({crow["N_sectors"]} sectors, {crow["N_stocks"]} stocks)')

# ---- Stage 3 summary ----
print(f'\n3. STAGE 3 -- Full Pooled Training ({len(train_paths):,} images)')
print(f'  Majority baseline: {pool_maj_bl*100:.2f}%')
print(f'  Direction baseline: {pool_dir_bl*100:.2f}%')
print(f'  VGG16:   Acc={pool_vgg_metrics["acc"]}%  '
      f'Dir={pool_vgg_metrics["dir_acc"]}%  '
      f'F1m={pool_vgg_metrics["f1_macro"]}%  '
      f'Cls={pool_vgg_metrics["n_pred_classes"]}/6')
print(f'  XGBoost: Acc={pool_xgb_metrics["acc"]}%  '
      f'Dir={pool_xgb_metrics["dir_acc"]}%  '
      f'F1m={pool_xgb_metrics["f1_macro"]}%  '
      f'Cls={pool_xgb_metrics["n_pred_classes"]}/6')

# Mode collapse
vgg_collapse = pool_vgg_metrics['n_pred_classes'] <= 2
xgb_collapse = pool_xgb_metrics['n_pred_classes'] <= 2
print(f'\n4. MODE COLLAPSE')
print(f'  Pooled VGG16:   {"COLLAPSED" if vgg_collapse else "RESOLVED"} '
      f'({pool_vgg_metrics["n_pred_classes"]} classes)')
print(f'  Pooled XGBoost: {"COLLAPSED" if xgb_collapse else "RESOLVED"} '
      f'({pool_xgb_metrics["n_pred_classes"]} classes)')

# Permutation test
print(f'\n5. PERMUTATION TEST')
print(f'  Real Acc: {perm_result["real_vgg"]["acc"]}%  '
      f'Perm Acc: {perm_result["perm_vgg"]["acc"]}%  '
      f'Gap: {perm_result["acc_gap_pp"]:+.2f} pp')
print(f'  Signal genuine: {perm_result["signal_genuine"]}  '
      f'(p={perm_result["p_real_vs_perm"]:.2e})')

# Specialist vs Generalist
print(f'\n6. SPECIALIST vs GENERALIST (avg delta across sectors)')
for tag in ['VGG', 'XGB']:
    ad = df_decomp[f'{tag} Acc delta'].dropna()
    dd = df_decomp[f'{tag} Dir delta'].dropna()
    winner = 'Generalist' if ad.mean() > 0 else 'Specialist'
    print(f'  {tag} Acc: {ad.mean():+.2f} pp  ({winner} wins on average)')
    winner_d = 'Generalist' if dd.mean() > 0 else 'Specialist'
    print(f'  {tag} Dir: {dd.mean():+.2f} pp  ({winner_d} wins on average)')

# ------------------------------------------------------------------
# Save consolidated final results
# ------------------------------------------------------------------
final = {
    'stage2': {
        'n_sectors': n_s2,
        'sector_results': {k: {kk: vv for kk, vv in v.items()
                                if kk != 'train_class_dist' and kk != 'test_class_dist'}
                           for k, v in stage2_results.items()
                           if v.get('status') == 'ok'},
    },
    'stage2_5': {
        'categories': cat_rows,
        'sector_volatility': sector_vol_info,
    },
    'stage3': {
        'n_train': len(train_paths),
        'n_test': len(test_paths),
        'majority_baseline': round(pool_maj_bl * 100, 2),
        'dir_baseline': round(pool_dir_bl * 100, 2),
        'vgg16': pool_vgg_metrics,
        'xgboost': pool_xgb_metrics,
        'permutation_test': perm_result,
    },
    'comparison': {
        'vgg_acc_delta_mean': round(float(df_decomp['VGG Acc delta'].dropna().mean()), 2),
        'vgg_dir_delta_mean': round(float(df_decomp['VGG Dir delta'].dropna().mean()), 2),
        'xgb_acc_delta_mean': round(float(df_decomp['XGB Acc delta'].dropna().mean()), 2),
        'xgb_dir_delta_mean': round(float(df_decomp['XGB Dir delta'].dropna().mean()), 2),
    },
}

with open(RESULTS_DIR / 'part8_final_results.json', 'w') as f:
    json.dump(final, f, indent=2, default=str)

print(f'\n{"=" * 70}')
print(f'Part 8 complete.')
print(f'All results saved to: {RESULTS_DIR}')
print(f'Key files:')
for fp in sorted(RESULTS_DIR.glob('*')):
    sz = fp.stat().st_size / 1024
    print(f'  {fp.name:>45s}  {sz:>8.1f} KB')
print(f'{"=" * 70}')